# omnicontext_gen_qlora_r8_step2000 — the 400-row benchmark at the selected QLoRA checkpoint (NF4 base)
Derived from `qlora_test_gen_r8` by swapping the data source only. Step 2000 is the argmax of GPT-4.1 val Overall over 500/1000/1500/2000 on the domain val slice; the checkpoint is not reselected here, since OmniContext measures what the domain-selected adapter does to general ability.

Transformer loaded NF4 before the adapter attaches, pipeline built once, one adapter, no swapping. The `Linear4bit` count and the adapter tensor count are asserted, because a silently unquantized base would be a bf16 run wearing the QLoRA label.

The base is quantized rather than bf16 because the adapter was fitted against `W + quantization error`: every arm in this thesis is scored on the configuration it was fitted to, and an NF4-trained adapter served on full-precision weights is the mismatch QA-LoRA exists to address.

Reads `omnicontext/manifest.jsonl` and `omnicontext/images/` at the pinned revision. Output: `gen/omnicontext/qlora_r8__step2000/{i:05d}.png`. Resumable.

Cost: 400 rows at roughly 20 s/img, near 2.3 h.


## 1. Repo, quantization backend

In [1]:
import os, subprocess, sys
REPO   = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# the version qlora_train_r8 and qlora_convert ran on; DiffusersBnBConfig.post_init() also reads it
subprocess.run([sys.executable,"-m","pip","install","-q","bitsandbytes==0.50.0"], check=True)

# peft's LoRA injection walks dispatch_torchao, which raises rather than returning False when torchao
# predates 0.16 (Colab ships 0.10). Same guard the trainer and converter use; no-op once gone.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print(_head[:7], "= repo commit")
assert _head == COMMIT, f"repo is at {_head[:7]}, expected {COMMIT[:7]}. Run: git -C {REPO} checkout {COMMIT}"
sys.path.insert(0, REPO)
os.chdir(REPO)

installed.
torchao removed
18e6f9d = repo commit


## 2. Drive, run config, adapter path

In [3]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "omnicontext"                # 400-row public benchmark, 8 subtasks x 50
RANK    = 8
STEP    = 2000                       # val argmax over 500/1000/1500/2000
STEPS   = [STEP]                       # kept as a list so the generation loop matches qlora_val_gen_r8
RUN     = f"qlora_r{RANK}"
EXP     = f"{DRIVE}/lora/experiments"  # qlora_convert writes alongside the LoRA adapters

lora = f"{EXP}/{RUN}/checkpoint-{STEP}/transformer_lora/pytorch_lora_weights.safetensors"
assert os.path.exists(lora), f"adapter missing, run qlora_convert first: {lora}"
print(f"{RUN}: selected checkpoint {STEP} present")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
qlora_r8: selected checkpoint 2000 present


## 3. Images, split files

In [4]:
import subprocess, json
OC       = f"{DRIVE}/omnicontext"
TAR      = f"{OC}/images.tar"
LOCALTAR = "/content/oc_images.tar"
LOCAL    = "/content/oc"
IMGS     = f"{LOCAL}/images"

rows     = [json.loads(l) for l in open(f"{OC}/manifest.jsonl", encoding="utf-8")]
REVISION = open(f"{OC}/revision.txt").read().strip()
EXPECTED = sum(len(r["input_images"]) for r in rows)

def n_files(d): return sum(len(f) for _,_,f in os.walk(d)) if os.path.isdir(d) else 0

if os.path.exists(TAR):
    subprocess.run(["cp", TAR, LOCALTAR], check=True)
else:
    # tar exits non-zero writing into the Drive FUSE mount; build locally, copy the one file up
    _r = subprocess.run(["tar", "-cf", LOCALTAR, "-C", OC, "images"], capture_output=True, text=True)
    assert os.path.exists(LOCALTAR), _r.stderr[-800:]
    subprocess.run(["cp", LOCALTAR, TAR], check=True)
if n_files(IMGS) < EXPECTED:
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["tar", "-xf", LOCALTAR, "-C", LOCAL], check=True)

assert n_files(IMGS) == EXPECTED, (n_files(IMGS), EXPECTED)
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows, all reference images present | revision {REVISION[:12]}")


omnicontext: 400 rows, all reference images present | revision 0f6aabbe8d34


## 4. Engine

In [5]:
from PIL import Image, ImageOps
import torch

# shipped default from inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

def generate_with(pipe, instruction, input_images=None, negative=NEG, seed=0, steps=50, text_g=5.0, image_g=2.0, size=512):
    gen = torch.Generator(device=accelerator.device).manual_seed(seed)
    res = pipe(prompt=instruction, input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=1024,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               negative_prompt=negative, num_images_per_prompt=1, generator=gen, output_type="pil",
               align_res=False)
    return res.images[0]

# no prompt lever in this arm: the adapter is the treatment, instruction and negative are the baseline
cfg = dict(instruction=lambda r: r["instruction"], negative=NEG)

## 5. Build and pass (resumable)

In [6]:
import torch, time, gc
from accelerate import Accelerator
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBnBConfig
import bitsandbytes as bnb

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16
N_LINEAR4BIT   = 309   # every nn.Linear in the transformer; SKIP_MODULES was [] in training
N_LORA_TENSORS = 304   # 38 attention blocks x 4 projections x (A, B)

# identical to the config qlora_train_r8 patched into train.py
QC = DiffusersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.bfloat16,
                        bnb_4bit_use_double_quant=True)

def build_pipe(step):
    LORA_DIR = f"{EXP}/{RUN}/checkpoint-{step}/transformer_lora"
    pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
    # mllm and vae are moved here while the transformer is still the stock bf16 one; bitsandbytes
    # places a 4-bit module on cuda itself and rejects a later .to(), so the swap comes after
    pipe = pipe.to(accelerator.device)
    pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(
        MODEL, subfolder="transformer", quantization_config=QC, torch_dtype=dtype)
    gc.collect(); torch.cuda.empty_cache()   # release the replaced bf16 transformer

    # a silently unquantized base would be a bf16 run wearing the QLoRA label, and the images would
    # look plausible; the count is the only thing that separates the two
    n4 = sum(1 for m in pipe.transformer.modules() if isinstance(m, bnb.nn.Linear4bit))
    assert n4 == N_LINEAR4BIT, f"base is not 4-bit: Linear4bit={n4}, expected {N_LINEAR4BIT}"

    pipe.transformer.load_lora_adapter(LORA_DIR, weight_name="pytorch_lora_weights.safetensors",
                                       prefix="transformer")
    n_lora = sum(1 for n, _ in pipe.transformer.named_parameters() if "lora_" in n)
    assert n_lora == N_LORA_TENSORS, f"adapter did not attach: {n_lora} tensors, expected {N_LORA_TENSORS}"

    dev = next(pipe.transformer.parameters()).device
    print(f"  step {step}: Linear4bit={n4}, lora tensors={n_lora}, transformer on {dev}")
    return pipe

def gen_step(step):
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    os.makedirs(GEN, exist_ok=True)
    have = {int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit()}
    todo = [i for i in range(len(rows)) if i not in have]
    print(f"\n=== step {step}: {len(have)} done, {len(todo)} to generate -> {GEN} ===")
    if not todo:
        return
    pipe = build_pipe(step)
    try:
        t0 = time.time()
        for k, i in enumerate(todo):
            r = rows[i]
            refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
            img = generate_with(pipe, cfg["instruction"](r), input_images=refs or None, negative=cfg["negative"])
            assert img.size == (512, 512), img.size
            img.save(f"{GEN}/{i:05d}.png")
            if (k+1) % 25 == 0:
                print(f"  {k+1}/{len(todo)}  ({(time.time()-t0)/(k+1):.1f}s/img)")
    finally:
        del pipe
        gc.collect(); torch.cuda.empty_cache()   # free VRAM before the next step's rebuild
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    print(f"  step {step}: {n}/{len(rows)} images")

for step in STEPS:
    gen_step(step)
print("\ndone")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")



=== step 2000: 0 done, 400 to generate -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/omnicontext/qlora_r8__step2000 ===


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  400/400  (31.9s/img)
  step 2000: 400/400 images

done


## 6. Manifest

In [7]:
import json, torch, datetime, bitsandbytes as bnb
for step in STEPS:
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else 0
    if n == 0: continue
    man = {"dataset": DATASET, "run": RUN, "step": step, "n_images": n,
           "dataset_revision": REVISION,
           "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "base": {"precision": "nf4", "double_quant": True, "compute_dtype": "bfloat16",
                    "linear4bit_modules": N_LINEAR4BIT, "bitsandbytes": bnb.__version__},
           "adapter": {"rank": RANK, "lora_tensors": N_LORA_TENSORS,
                       "path": f"{EXP}/{RUN}/checkpoint-{step}/transformer_lora"},
           "sampler": {"steps": 50, "text_guidance_scale": 5.0, "image_guidance_scale": 2.0,
                       "cfg_range": [0.0, 1.0], "max_sequence_length": 1024, "seed": 0,
                       "size": 512, "align_res": False},
           "negative_prompt": NEG,
           "selection": "argmax val Overall over 500/1000/1500/2000; post-warmup argmax agrees",
           "written": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(man, open(f"{GEN}/manifest.json", "w"), indent=2)
print("manifests written")

manifests written
